In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Варіант 1 (Київ)
np.random.seed(1)
city = "Київ"
base_temp = 9.5          # середньорічна температура, °C
amplitude = 14           # сезонна амплітуда, °C
base_consumption = 5200  # базове споживання, МВт·год
k = 55                   # нахил (МВт·год на кожен градус)
noise_sd = 80            # рівень шуму споживання, МВт·год

months = np.arange(1, 13)
temp = base_temp + amplitude * np.cos((months - 7) / 12 * 2 * np.pi)
temp = np.round(temp + np.random.normal(0, 0.5, size=12), 1)

# Чим температура нижча за середньорічну, тим більше йде на опалення
consumption = base_consumption - k * (temp - base_temp) + np.random.normal(0, noise_sd, size=12)
consumption = np.round(consumption, 0)

city_data = pd.DataFrame({
    "місяць": months, 
    "температура": temp, 
    "споживання_МВтгод": consumption,
})

print(city_data)

In [ ]:
# Підбір лінійної моделі за допомогою np.polyfit
b, a = np.polyfit(city_data["температура"], city_data["споживання_МВтгод"], 1)

print(f"Підібране рівняння: споживання ≈ {a:.1f} + ({b:.1f}) · температура")

# Додаємо передбачення до DataFrame
city_data["передбачення"] = a + b * city_data["температура"]
print(city_data)

In [ ]:
# Обчислення залишків
residuals = city_data["споживання_МВтгод"] - city_data["передбачення"]
city_data["залишок"] = residuals

# Таблиця порівняння
print(city_data[["місяць", "температура", "споживання_МВтгод", "передбачення", "залишок"]])

# Обчислення R²
ss_res = (residuals ** 2).sum()
ss_tot = ((city_data["споживання_МВтгод"] - city_data["споживання_МВтгод"].mean()) ** 2).sum()
r_squared = 1 - ss_res / ss_tot
print(f"\nКоефіцієнт детермінації R² = {r_squared:.4f}")

# Побудова графіка залишків
plt.figure(figsize=(9, 4))
plt.plot(city_data["місяць"], residuals, marker='o', color='purple', linestyle='-')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Місяць року")
plt.ylabel("Залишок (Смотрено - Передбачено)")
plt.title("Графік залишків по місяцях")
plt.grid(True)
plt.show()

## Контрольні питання

1. **Як чітко розмежувати верифікацію й валідацію на прикладі власної моделі?**
   * **Верифікація**: перевірка, чи правильно розраховується формула $a + b \cdot T$ програмним шляхом (тестування коду).
   * **Валідація**: перевірка працездатності моделі на нових незалежних даних (наприклад, даних споживання електроенергії за наступний рік).

2. **За якими незалежними ознаками класифікується модель і чому вони не суперечать одна одній?**
   Модель класифікується за такими осями: стохастична (наявність шуму), статична (відсутність рекурентності в часі) та описово-прогнозна (не балансова/оптимізаційна). Вони відповідають на різні питання: характер урахування невизначеності, часову структуру взаємозв'язку та мету моделювання.

3. **Що показує $R^2$ і чи доводить він придатність моделі для екстраполяції?**
   $R^2$ показує частку поясненої дисперсії у межах спостереженого діапазону. Високе значення $R^2$ **не гарантує** адекватності при екстраполяції за межі вибірки, оскільки за межами зібраних температур зв'язок може перестати бути лінійним.

4. **Чому структура залишків інформативніша за сумарну величину відхилень?**
   Сумарні показники можуть маскувати систематичні помилки (наприклад, недооцінку взимку і переоцінку влітку, які взаємно гасяться). Наявність або відсутність структури на графіку залишків одразу показує, чи правильно обрана базова математична форма моделі.